# Gold — Carga no PostgreSQL

Lê os 4 marts Delta Lake e carrega no schema `finops_gold` do PostgreSQL para consumo pelo Metabase.

**Tabelas carregadas:**
- `finops_gold.monthly_cost_by_team`
- `finops_gold.top_resources`
- `finops_gold.cost_trend_by_provider`
- `finops_gold.cost_by_environment`

**Modo:** `overwrite` (recalcula sempre, garante idempotência)

In [ ]:
# parameters
execution_date = "2025-06-15"
year = 2025
month = 6
day = 15


In [ ]:
import sys
import os

notebooks_root = next(
    (p for p in ["/opt/airflow/notebooks", "/opt/spark/notebooks"]
     if __import__("os").path.exists(p)),
    "/opt/spark/notebooks",
)
if notebooks_root not in sys.path:
    sys.path.insert(0, notebooks_root)

from utils.spark_session import create_spark_session, postgres_jdbc_url, postgres_props
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType, StringType, DecimalType, TimestampType, DateType
)


In [ ]:
import os

spark = create_spark_session("gold_load_postgres")

jdbc_url = postgres_jdbc_url()
jdbc_props = {
    "user": os.getenv("POSTGRES_USER", "finops_user"),
    "password": os.getenv("POSTGRES_PASSWORD", "finops_pass"),
    "driver": "org.postgresql.Driver",
}

marts = {
    "monthly_cost_by_team":   "s3a://datalake/gold/monthly_cost_by_team/",
    "top_resources":           "s3a://datalake/gold/top_resources/",
    "cost_trend_by_provider":  "s3a://datalake/gold/cost_trend_by_provider/",
    "cost_by_environment":     "s3a://datalake/gold/cost_by_environment/",
}

for table_name, delta_path in marts.items():
    df = spark.read.format("delta").load(delta_path).drop("_processed_at")
    row_count = df.count()

    (
        df.write
        .format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", f"finops_gold.{table_name}")
        .options(**jdbc_props)
        .mode("overwrite")
        .save()
    )
    print(f"  ✓ finops_gold.{table_name}: {row_count:,} linhas carregadas")

print("\n[gold_load_postgres] Carga concluída. Metabase pode consultar finops_gold.*")
spark.stop()
